# From-Scratch Neural Network for Detecting Protected Information

Goal: Build a simple feedforward NN using only NumPy to classify text snippets as containing:
- PII (Personally Identifiable Information)
- HIPAA-related (health info patterns)
- PCI (payment card info)
- Other protected
- None

Approach:
- Manual forward + backpropagation
- No PyTorch/TensorFlow/autograd
- Synthetic or small public data
- Focus: correctness of math & gradients

Sections to follow:
1. Data preparation
2. Model architecture
3. Forward pass
4. Loss
5. Backward pass
6. Optimizer
7. Training loop
8. Evaluation

In [ ]:
import numpy as np # only math engine
from sklearn.model_selection import train_test_split # quick & reliable data splitting
from sklearn.feature_extraction.text import CountVectorizer  # only for simple vectorization easiest way to turn text → fixed-size vectors (bag-of-words) without writing your own tokenizer from zero
import re  # for regex-based feature help later
import matplotlib.pyplot as plt  # you'll need this eventually for loss plots
import pandas as pd  # data manipulation and exporting to csv/excel
from datasets import load_dataset # for importing HuggingFace datasets
from google.colab import userdata # for using HuggingFace token
userdata.get('HF_TOKEN')
import json #for saving jsonl files
from pathlib import Path

# Optional: set random seed for reproducibility
randseed = np.random.seed(42)

print("NumPy version:", np.__version__)
print("Setup complete – ready to build data and model.")
print("NP random seed: ", np.random.seed(42))

NumPy version: 2.0.2
Setup complete – ready to build data and model.
NP random seed:  None


Data exploration with ai4privacy/pii-masking-300k


In [ ]:
dataset = load_dataset(
    "ai4privacy/pii-masking-300k",
    data_files={
        "train": "data/train/1english_openpii_*.jsonl",
        "validation": "data/validation/1english_openpii_*.jsonl"
    }
)
#print(dataset)  # Confirm only English loaded
#print(dataset["train"][0])  # Inspect first example

Convert to pandas dataframe


In [ ]:
df = dataset["train"].to_pandas()

In [ ]:
df.head()

,source_text,target_text,privacy_mask,span_labels,mbert_text_tokens,mbert_bio_labels,id,language,set
0,Subject: Group Messaging for Admissions Proces...,Subject: Group Messaging for Admissions Proces...,"[{'value': 'wynqvrh053', 'start': 287, 'end': ...","[[440, 453, ""USERNAME""], [430, 437, ""TIME""], [...","[Sub, ##ject, :, Group, Mess, ##aging, for, Ad...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",40767A,English,train
1,- Meeting at 2:33 PM\n- N23 - Meeting at 11:29...,- Meeting at [TIME]\n- [USERNAME] - Meeting at...,"[{'value': '2:33 PM', 'start': 13, 'end': 20, ...","[[74, 81, ""TIME""], [50, 60, ""USERNAME""], [40, ...","[-, Meeting, at, 2, :, 33, PM, -, N, ##23, -, ...","[O, O, O, B-TIME, I-TIME, I-TIME, I-TIME, O, O...",40767B,English,train
2,Subject: Admission Notification - Great Britai...,Subject: Admission Notification - Great Britai...,"[{'value': '5:24am', 'start': 263, 'end': 269,...","[[395, 407, ""SOCIALNUMBER""], [358, 375, ""EMAIL...","[Sub, ##ject, :, Ad, ##mission, Not, ##ificati...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",40768A,English,train
3,Card: KB90324ER\n Country: GB\n Building: ...,Card: [IDCARD]\n Country: [COUNTRY]\n Buil...,"[{'value': 'KB90324ER', 'start': 6, 'end': 15,...","[[390, 393, ""STATE""], [368, 378, ""CITY""], [346...","[Card, :, KB, ##90, ##32, ##4, ##ER, \, n, Cou...","[O, O, B-IDCARD, I-IDCARD, I-IDCARD, I-IDCARD,...",40768B,English,train
4,"N, WA14 5RW\n Password: r]iD1#8\n\n...and so...","N, WA14 5RW\n Password: [PASS]\n\n...and so ...","[{'value': 'r]iD1#8', 'start': 26, 'end': 33, ...","[[336, 352, ""DATE""], [26, 33, ""PASS""]]","[N, ,, W, ##A, ##14, 5, ##R, ##W, \, n, Pass, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-PASS...",40768C,English,train


In [ ]:
# Sample first — only 1000 rows
sample_df = dataset["train"].shuffle(seed=42).select(range(1000)).to_pandas()

# Now explore the small sample — fast
print(sample_df.info())
print("\nFirst 5 rows:")
print(sample_df.head(5))
print("\nShape:", sample_df.shape)
print("\nColumns:", sample_df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   source_text        1000 non-null   object
 1   target_text        1000 non-null   object
 2   privacy_mask       1000 non-null   object
 3   span_labels        1000 non-null   object
 4   mbert_text_tokens  1000 non-null   object
 5   mbert_bio_labels   1000 non-null   object
 6   id                 1000 non-null   object
 7   language           1000 non-null   object
 8   set                1000 non-null   object
dtypes: object(9)
memory usage: 70.4+ KB
None

First 5 rows:
                                         source_text  \
0  "Welcome to the Real-time Website Chat for Ear...   
1  \n- Country: DE  \n- Building: 4  \n- Street: ...   
2  {\n  "Individuals": {\n    "Person1": {\n     ...   
3   {\n          "Framework": "Established",\n   ...   
4            <passport>573201265</pas

In [ ]:
# Quick overview
sample_df.info()                          # column names, types, non-null counts
sample_df.head(5)                         # first 5 rows (shows source_text, privacy_mask, etc.)
sample_df.tail(3)
sample_df.describe()                      # basic stats (mostly useful for numeric if any)
sample_df.shape                           # (rows, columns)
Anubads_From_Scratch_Neural_Network
# Column names (features)
print(sample_df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   source_text        1000 non-null   object
 1   target_text        1000 non-null   object
 2   privacy_mask       1000 non-null   object
 3   span_labels        1000 non-null   object
 4   mbert_text_tokens  1000 non-null   object
 5   mbert_bio_labels   1000 non-null   object
 6   id                 1000 non-null   object
 7   language           1000 non-null   object
 8   set                1000 non-null   object
dtypes: object(9)
memory usage: 70.4+ KB
['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']


Balance Check

In [ ]:
# Balance check: count rows with empty vs non-empty privacy_mask

from collections import Counter

# We'll count manually for speed
empty_count = 0
non_empty_count = 0
multi_entity_count = 0  # optional: how many have >1 PII entity

for example in dataset["train"]:
    mask = example["privacy_mask"]
    if len(mask) == 0:
        empty_count += 1
    else:
        non_empty_count += 1
        if len(mask) > 1:
            multi_entity_count += 1

total = empty_count + non_empty_count

print(f"Total rows: {total}")
print(f"Empty privacy_mask (None / no PII): {empty_count} ({empty_count / total * 100:.1f}%)")
print(f"Has PII (non-empty): {non_empty_count} ({non_empty_count / total * 100:.1f}%)")
print(f"Of those with PII, rows with >1 entity: {multi_entity_count}")

Total rows: 29908
Empty privacy_mask (None / no PII): 3482 (11.6%)
Has PII (non-empty): 26426 (88.4%)
Of those with PII, rows with >1 entity: 24525


Save JSON files of dataset partitions

In [ ]:
# Step 1: Get indices for non-PII and PII rows
non_pii_indices = []
pii_indices = []

for idx, example in enumerate(dataset["train"]):
    if len(example["privacy_mask"]) == 0:
        non_pii_indices.append(idx)
    else:
        pii_indices.append(idx)

print(f"Non-PII rows: {len(non_pii_indices)}")
print(f"PII rows: {len(pii_indices)}")

# Step 2: Shuffle only PII indices
np.random.shuffle(pii_indices)

# Step 3: Split shuffled PII into 5 equal chunks of 3482
chunk_size = 3482
pii_chunks = []
for i in range(5):
    start = i * chunk_size
    end = start + chunk_size
    chunk = pii_indices[start:end]
    pii_chunks.append(chunk)

print(f"Each chunk size: {len(pii_chunks[0])}")  # should be 3482

# Step 4: Save non-PII as one file
non_pii_path = Path("/content/non_pii_fixed.jsonl")
with open(non_pii_path, "w", encoding="utf-8") as f:
    for idx in non_pii_indices:
        example = dataset["train"][idx]
        f.write(json.dumps(example) + "\n")

print(f"Saved non-PII: {non_pii_path}")

# Step 5: Save each PII chunk
for cycle_num, chunk_indices in enumerate(pii_chunks, 1):
    file_path = Path(f"/content/pii_partition_{cycle_num}.jsonl")
    with open(file_path, "w", encoding="utf-8") as f:
        for idx in chunk_indices:
            example = dataset["train"][idx]
            f.write(json.dumps(example) + "\n")

    print(f"Saved cycle {cycle_num}: {file_path}")

print("\nAll files saved! Check the Files panel on the left.")

Non-PII rows: 3482
PII rows: 26426
Each chunk size: 3482
Saved non-PII: /content/non_pii_fixed.jsonl
Saved cycle 1: /content/pii_partition_1.jsonl
Saved cycle 2: /content/pii_partition_2.jsonl
Saved cycle 3: /content/pii_partition_3.jsonl
Saved cycle 4: /content/pii_partition_4.jsonl
Saved cycle 5: /content/pii_partition_5.jsonl

All files saved! Check the Files panel on the left.
